# 03 · Ablation: fine-tuning WITHOUT the decline examples (Colab, T4 GPU)

**Question:** the fine-tuned model declines 12% of held-out-item questions and 20% of casual questions.
Is that caused by the ~4% out-of-scope "decline" examples in the fine-tuning data?

**Experiment:** repeat fine-tuning with *everything identical* (same pretrained start, seed, steps, learning rate)
except that the `out_of_scope` examples are removed. Output: `MyDrive/mini-gpt/sft_no_decline/`.

`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all` (about 1–2 minutes). 📸 = screenshot for the report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/mini-gpt'
PRETRAINED = f'{DRIVE}/pretrain_run2/ckpt_best.pt'
RUN_DIR = f'{DRIVE}/sft_no_decline'
RESULTS = f'{DRIVE}/results'
assert os.path.exists(PRETRAINED), f'pretrained checkpoint not found: {PRETRAINED}'
os.makedirs(RUN_DIR, exist_ok=True)
for sub in ('figures', 'logs', 'samples'):
    os.makedirs(f'{RESULTS}/{sub}', exist_ok=True)

REPO = 'https://github.com/Shayan047/mini-gpt.git'
if os.path.exists('/content/mini-gpt'):
    !git -C /content/mini-gpt pull
else:
    !git clone {REPO} /content/mini-gpt
%cd /content/mini-gpt
!git log --oneline -1
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
# 📸 Ablation fine-tuning log (only difference from notebook 02: --exclude-categories out_of_scope)
!python -m src.finetune --init "{PRETRAINED}" --out-dir "{RUN_DIR}" --exclude-categories out_of_scope

In [ ]:
!python -m src.plot_sft --log "{RUN_DIR}/log.csv" --out "{RESULTS}/figures/sft_no_decline_loss.png"
from IPython.display import Image, display
display(Image(f'{RESULTS}/figures/sft_no_decline_loss.png'))
import shutil
for f in ('log.csv', 'train_log.txt', 'summary.json'):
    shutil.copy(f'{RUN_DIR}/{f}', f'{RESULTS}/logs/sft_no_decline_{f}')
print(open(f'{RUN_DIR}/summary.json').read())
print('Download', f'{RUN_DIR}/ckpt_best.pt', '-> repo checkpoints/sft_no_decline/ckpt_best.pt')